In [ ]:
import pandas as pd
from datetime import datetime, timedelta
from fuzzywuzzy import process

from aiogram import Bot, Dispatcher
from aiogram.types import Message
from aiogram.filters import CommandStart

BOT_TOKEN = "8719214919:AAHQV9OPhkKJZsuPeiLGP9vyLLTlLM1KYGg"

bot = Bot(token=BOT_TOKEN)
dp = Dispatcher()

FILE = "matches.xlsx"

# ====================
# قراءة الاكسيل
# ====================

def load_data():
    df = pd.read_excel(FILE)

    df.columns = df.columns.str.lower()

    df["team1"] = df["team1"].astype(str)
    df["team2"] = df["team2"].astype(str)
    df["date"] = df["date"].astype(str)
    df["time"] = df["time"].astype(str)

    return df

# ====================
# تحويل 12-06 الى 2026-06-12
# ====================

def normalize_date(text):
    text = text.replace("/", "-")

    parts = text.split("-")

    if len(parts) == 2:
        day, month = parts
        year = datetime.now().year

        return f"{year}-{month.zfill(2)}-{day.zfill(2)}"

    return text

# ====================
# Start
# ====================

@dp.message(CommandStart())
async def start(message: Message):

    await message.answer(
        "⚽ بوت الماتشات جاهز\n\n"
        "اكتب:\n"
        "اليوم\n"
        "today\n"
        "بكره\n"
        "غدا\n"
        "tomorrow\n"
        "مصر\n"
        "Egypt\n"
        "12-06\n"
        "2026-06-12"
    )

# ====================
# أي رسالة
# ====================

@dp.message()
async def search(message: Message):

    text = message.text.strip().lower()

    df = load_data()

    # ====================
    # اليوم
    # ====================

    if text in ["today", "اليوم"]:

        today = datetime.now().strftime("%Y-%m-%d")

        result = df[df["date"] == today]

        if result.empty:
            return await message.answer("❌ لا توجد مباريات اليوم")

        reply = "📅 مباريات اليوم\n\n"

        for _, row in result.iterrows():

            reply += (
                f"⚽ {row['team1']} 🆚 {row['team2']}\n"
                f"🕒 {row['time']}\n\n"
            )

        return await message.answer(reply)

    # ====================
    # بكرة
    # ====================

    if text in ["tomorrow", "بكره", "بكرة", "غدا", "غداً", "ماتشات بكره", "ماتشات بكرة"]:

        tomorrow = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")

        result = df[df["date"] == tomorrow]

        if result.empty:
            return await message.answer("❌ لا توجد مباريات غداً")

        reply = "📅 مباريات الغد\n\n"

        for _, row in result.iterrows():

            reply += (
                f"⚽ {row['team1']} 🆚 {row['team2']}\n"
                f"🕒 {row['time']}\n\n"
            )

        return await message.answer(reply)

    # ====================
    # تاريخ
    # ====================

    if "-" in text or "/" in text:

        date_value = normalize_date(text)

        result = df[df["date"] == date_value]

        if result.empty:
            return await message.answer("❌ لا توجد مباريات")

        reply = f"📅 مباريات {date_value}\n\n"

        for _, row in result.iterrows():

            reply += (
                f"⚽ {row['team1']} 🆚 {row['team2']}\n"
                f"🕒 {row['time']}\n\n"
            )

        return await message.answer(reply)

    # ====================
    # فريق
    # ====================

    all_teams = list(set(
        df["team1"].str.lower().tolist()
        +
        df["team2"].str.lower().tolist()
    ))

    match, score = process.extractOne(text, all_teams)

    if score < 40:
        return await message.answer("❌ لم يتم العثور على فريق")

    result = df[
        (df["team1"].str.lower() == match)
        |
        (df["team2"].str.lower() == match)
    ]

    if result.empty:
        return await message.answer("❌ لا توجد مباريات")

    reply = f"⚽ مباريات {match}\n\n"

    for _, row in result.iterrows():

        reply += (
            f"📅 {row['date']}\n"
            f"⚽ {row['team1']} 🆚 {row['team2']}\n"
            f"🕒 {row['time']}\n\n"
        )

    await message.answer(reply)

# ====================
# تشغيل البوت
# ====================

import asyncio

async def main():

    print("🤖 Bot Running...")

    await dp.start_polling(bot)

await main()